# pw02 — 오픈소스 센싱/ISAC 도구 지도

> ⚠ **이 노트북은 생성물이다. 수정은 `prior_work/src/make_pw02.py` 에서** 하고 재실행할 것.
> 모든 사실·라이선스는 `prior_work/outputs/prior_work.json`(직접 GitHub/arXiv 확인) 에서 주입한다.

질문: **이미 존재하는 오픈소스가 우리 조각(패시브 바이스태틱 + SBR+PO 드론 RCS + ECA/CAF/CFAR)을 대신 해 주는가?** 답을 미리 말하면 — **통째로 해 주는 도구는 없지만, 조각별로 빌려 쓸 좋은 것이 있다**(OpenISAC·RadarSimPy). 무엇을 어떻게 채택할지 정한다.

## §1. 능력 매트릭스 (한눈에)

| 도구 | 라이선스 | 조명 파형 | 메쉬 RCS | 검출체인(CFAR) | 패시브 바이스태틱 | 채택 |
|---|---|---|---|---|---|---|
| NIST 5GNRad | NIST-developed | 5G NR | — | ✅ | — | HIGH |
| RadarSimPy | GPLv3 | — | ✅ | ✅ | — | HIGH |
| OpenISAC | 오픈소스 | 연속 OFDM | — | ✅ | ✅ | ⭐ HIGH |
| MATLAB 5G/Radar/Phased Array Toolboxes | 상용 | 5G NR | — | ✅ | — | MEDIUM |
| ns3sionna | 오픈소스 | — | — | — | — | LOW — 네트워크/MAC 레벨. 우리 물리레이어 패시브 레이더와 층이 다름 |
| ms-van3t-rt | 오픈소스 | — | — | — | — | LOW — V2X 통신 전용 |

> **읽는 법.** 어떤 한 도구도 6칸을 다 채우지 못한다 — 이게 우리가 스택을 **직접 조립한** 이유다. 다만 **RadarSimPy 는 메쉬 RCS**, **OpenISAC 은 X410 바이스태틱**, **NIST 5GNRad 은 검출체인 아키텍처**를 각각 잘 해 준다.

---
## §2. 채택도 HIGH — 실제로 쓸 것

### 🔎직접확인  NIST 5GNRad  —  채택도 판정: **HIGH**
- **저장소·출처**: github.com/usnistgov/5GNRad (Steve Blandino, NIST CTL)  ([1](https://github.com/usnistgov/5GNRad))
- **라이선스**: NIST-developed (US Gov, 사실상 퍼블릭 도메인)
- **무엇을 하나**: 5G NR ISAC 링크레벨 시뮬레이터 — 표준호환 NR 파형·채널추정·거리도플러·**CFAR**·각도추정·3D 위치
- **표적 산란 처리**: c) **RCS 점표적** 주입 + target/background 채널 분리(h=h_bg+h_target)
- **검출체인?** ✅ 풀 검출체인(range-Doppler·CFAR·clustering·위치/속도 추정·TP/FP/FN)
- **우리 프로젝트 채택**: HIGH(아키텍처 참조) — h=h_bg+h_target 분리가 **우리와 동일**하다는 강력한 방증. 단 능동/PRS 기반이라 패시브 reference/surveillance 구조는 우리가 추가해야 함

### 🔎직접확인  RadarSimPy  —  채택도 판정: **HIGH**
- **저장소·출처**: github.com/radarsimx/radarsimpy  ([1](https://github.com/radarsimx/radarsimpy) · [2](https://github.com/radarsimx/radarsimpy/blob/master/LICENSE))
- **라이선스**: GPLv3 (완전 오픈)
- **무엇을 하나**: Python+C++ 레이더 시뮬 — 점표적 + **3D STL 메쉬 RCS**, 마이크로도플러/터빈, Swerling, 거리도플러·DoA·CFAR
- **표적 산란 처리**: 3D 메쉬 RCS 직접 계산(광선추적 기반). 다만 5G/WiFi 통신 파형 스택은 없음
- **검출체인?** ✅ 거리도플러·CFAR·DoA
- **우리 프로젝트 채택**: HIGH(검증 오라클) — 우리 SBR+PO 드론 RCS·프로펠러 마이크로도플러를 **독립 도구로 교차검증**하는 데 이상적. GPLv3 라 자유 사용

### 🔎직접확인  OpenISAC  —  채택도 판정: **⭐ HIGH**
- **저장소·출처**: github.com/zhouzhiwen2000/OpenISAC · arXiv:2601.03535 (2026-01)  ([1](https://arxiv.org/abs/2601.03535) · [2](https://github.com/zhouzhiwen2000/OpenISAC))
- **라이선스**: 오픈소스
- **무엇을 하나**: 실시간 OFDM-ISAC 실험 플랫폼 — 모노+**바이스태틱** 지연도플러, **OTA 동기(유선없이 바이스태틱)**, USRP B200~**X400 계열**, C++PHY+Python센싱, 마이크로도플러 추출 검증
- **표적 산란 처리**: OTA 실측(시뮬 아님)
- **검출체인?** ✅ 지연도플러 검출, 마이크로도플러
- **우리 프로젝트 채택**: ⭐ HIGH(실측 단계) — **X410 지원 + 바이스태틱 OTA 동기**가 사용자 하드웨어 계획과 직결. sim→real 다리. 단 연속 커스텀 OFDM 이라 표준 WiFi/LTE/5G 파형은 별도

> 🔑 **세 도구의 역할 분담(우리 프로젝트에서).**
> - **NIST 5GNRad** — *아키텍처 참조*. `h = h_background + h_target`(표적/배경 채널 분리)이 우리 report12 구조와 **동일**하다는 강력한 방증. 검출체인(range-Doppler·CFAR·clustering) 설계를 대조한다. (능동/PRS 기반이라 코드 이식은 아님.)
> - **RadarSimPy** (GPLv3) — *검증 오라클*. 3D STL 메쉬에서 RCS·프로펠러 마이크로도플러를 **독립 도구로 다시 계산**해 우리 SBR+PO(report07/08)와 대조한다. 완전 오픈이라 자유 사용.
> - **OpenISAC** — *실측 다리*. **USRP X410 + 바이스태틱 OTA 동기**가 사용자 하드웨어 계획과 직결. sim→real 단계에서 실제 OTA 실험 골격으로 채택.

---
## §3. 채택도 MEDIUM / LOW — 참조 또는 불채택

### ⚠미검증단서  MATLAB 5G/Radar/Phased Array Toolboxes  —  채택도 판정: **MEDIUM**
- **저장소·출처**: MathWorks 공식 (유료)  ([1](https://www.mathworks.com/help/5g/ug/integrated-sensing-and-communication-using-5g-waveform.html))
- **라이선스**: 상용(학교 라이선스 흔함)
- **무엇을 하나**: 가장 완성형 ISAC 예제 — 5G NR PDSCH 전송→DM-RS 채널추정→이동 UAV range/angle→CFAR·추적. 바이스태틱 레이더 예제도 제공
- **표적 산란 처리**: c) RCS 상수 입력(phased.RadarTarget MeanRCS=)
- **검출체인?** ✅ 전부(CFAR·추적)
- **우리 프로젝트 채택**: MEDIUM(참조·1차 baseline) — 구조가 우리와 동일함을 확인하는 표준 레퍼런스. 파이썬 프로젝트라 코드 이식은 안 함

### 📄단일출처  ns3sionna  —  채택도 판정: **LOW — 네트워크/MAC 레벨. 우리 물리레이어 패시브 레이더와 층이 다름**
- **저장소·출처**: github.com/tkn-tub/ns3sionna (Zubow·Rösler·Dressler, TU Berlin) · arXiv:2412.20524  ([1](https://github.com/tkn-tub/ns3sionna) · [2](https://arxiv.org/abs/2412.20524))
- **라이선스**: 오픈소스
- **무엇을 하나**: ns-3 + Sionna RT 채널결합(SionnaPropagationLoss/Delay/Mobility 모델). CFR 태그로 CSI 노출
- **표적 산란 처리**: Sionna 확산(채널만)
- **검출체인?** 없음 — CSI 내보내기만, 레이더/RCS/CFAR 전무
- **우리 프로젝트 채택**: LOW — 네트워크/MAC 레벨. 우리 물리레이어 패시브 레이더와 층이 다름

### 📄단일출처  ms-van3t-rt  —  채택도 판정: **LOW — V2X 통신 전용**
- **저장소·출처**: github.com/robpegurri/ms-van3t-rt · arXiv:2501.00372 (PoliMi + NVIDIA J.Hoydis)  ([1](https://arxiv.org/pdf/2501.00372))
- **라이선스**: 오픈소스
- **무엇을 하나**: ns-3 V2X 풀스택 + Sionna RT 채널(디지털 네트워크 트윈)
- **표적 산란 처리**: Sionna 확산(채널만)
- **검출체인?** 없음 — ISAC/레이더/RCS/CFAR 전무
- **우리 프로젝트 채택**: LOW — V2X 통신 전용

> **왜 ns3sionna·ms-van3t 는 불채택인가.** 둘 다 Sionna 를 ns-3 **네트워크/MAC 층**에 붙이는 채널결합이다 — CSI 를 내보낼 뿐 레이더 표적산란·RCS·CFAR 이 전무하다. 우리 물리레이어 패시브 레이더와 다른 층의 도구다.

In [ ]:
# 도구별 채택 판정 — prior_work.json 에서
import json
J = json.load(open('outputs/prior_work.json', encoding='utf-8'))
for t in J['tools']:
    tag = t['adopt'].split('(')[0].strip()
    print(f"{tag:8s} | {t['name']:26s} | {t['license'][:22]:22s} | {t['repo'][:40]}")

---
## §4. 정리 — 채택 계획

OpenISAC→X410 실측(바이스태틱 OTA 동기) · RadarSimPy(GPLv3)→SBR+PO RCS·마이크로도플러 독립 교차검증 오라클 · NIST 5GNRad→검출체인 아키텍처 참조 · ns3sionna/ms-van3t→불채택(네트워크층).

요컨대 **시뮬 단계**는 우리 스택(Sionna 챔버 + SBR+PO + 자체 검출)을 유지하되 **RadarSimPy 로 RCS·마이크로도플러를 교차검증**하고, **실측 단계**는 **OpenISAC+X410** 골격을 빌려 OTA 바이스태틱을 실현한다. **NIST 5GNRad** 는 검출체인 설계의 표준 레퍼런스로 둔다.

> **다음** → [pw03 — 우리 방법의 위치와 선행 방법론 수용](pw03_positioning.ipynb).